# NOTES2 — Change Log (Fable 5 session)

**Author of changes:** Claude `Fable 5` (claude-fable-5), driven by Hari.
**Date:** 2026-06-11.

This notebook is a **reference of every change made in the Fable 5 session**, kept separate from `NOTES.ipynb` (the original project documentation) so you can see exactly what was touched and why. Nothing here changes the science described in `NOTES.ipynb` — it records the engineering edits on top of it.

**Contents:**
1. TL;DR
2. Was the program actually broken? (diagnosis)
3. Change 1 — settings.local.json permission fix
4. Change 2 — real RSA via Shrake-Rupley (the big one)
5. Change 3 — half-sphere exposure features
6. Change 4 — housekeeping (config + encoder counts)
7. File-by-file summary
8. Results: before vs after (quick-mode + full tuned Optuna)
9. Candidate structural features to add next
10. Do we need more protein structures?
11. Other (non-structural) features we could add
12. ENaC comparison (sibling VEP-Enac project)

## 1. TL;DR

- The pipeline was **not broken** — it already ran end-to-end and reproduced the documented baselines (LightGBM F1 = 0.654 without structural, 0.666 with). "Stitching" was really about finding *dead weight* in the features.
- The structural feature **RSA was dead**: all 351 values were a constant `1.0` because real RSA needed the `mkdssp` binary, which is not available on Windows, so it was imputed.
- **Fix:** compute real solvent accessibility with **BioPython's Shrake-Rupley** algorithm (`Bio.PDB.SASA.ShrakeRupley`), which runs directly on the CIF coordinates with **no external binary**. RSA went from 1 unique value → 137 unique values.
- **Added 2 features:** half-sphere exposure (`hse_up`, `hse_down`) as a complementary burial measure.
- SASA is computed over the **full pentamer**, so residues buried at subunit–subunit interfaces are correctly scored as buried (biologically right for a nAChR assembly).
- Structural feature group: **6 → 8 features**. Total with structural: **50 → 52**.

## 2. Was the program actually broken?

No. Running both paths reproduced the numbers in `NOTES.ipynb` section 7:

```
python scripts/run_experiment.py --quick --no-structural
  logistic_regression : F1=0.6483   random_forest : F1=0.6350
  svm_rbf             : F1=0.5998   lightgbm      : F1=0.6540

python scripts/run_experiment.py --quick   (with structural, OLD code)
  lightgbm            : F1=0.6658
```

Data loading, all four feature groups, the encoder, the model registry, Optuna nested-CV, and result saving were all wired correctly. The only runtime noise was harmless sklearn *"X does not have valid feature names"* warnings (cosmetic; left as-is).

**Diagnosis that mattered** (per-column structural stats, OLD code):

| feature | unique values | note |
|---|---|---|
| `rsa` | **1** | dead — constant 1.0 (mkdssp missing) |
| `bfactor` | 194 | working |
| `dssp_helix/sheet/coil` | 2 each | working |
| `cbeta_density` | 23 | working |

267/351 mutations map to a PDB residue; 84/351 are imputed (6 subunits have no structure + some unresolved positions).

## 3. Change 1 — `.claude/settings.local.json` permission fix

**File:** `Variant-Effect-Predictor-for-nAChRs/.claude/settings.local.json`

Removed an invalid permission rule flagged by `/doctor`. A bare `mcp__*` wildcard is not allowed in an **allow** list (only in deny/ask). No MCP servers are configured in this project, so it was simply deleted.

```diff
 "allow": [
   ...
   "TodoWrite(*)",
-  "mcp__*"
 ]
```

(Unrelated to the model code — listed here for completeness.)

## 4. Change 2 — real RSA via Shrake-Rupley (the big one)

**File:** `vep_nachr/features/structural.py`

### What & why
RSA (relative solvent accessibility = how exposed vs buried a residue is) is, per `NOTES.ipynb` §4.4, *"potentially the most informative structural feature."* It was dead because the old code only got ASA from `mkdssp`, which doesn't run on Windows, and otherwise imputed `1.0`.

**Shrake-Rupley** (the classic rolling-probe SASA algorithm) ships inside BioPython (`Bio.PDB.SASA.ShrakeRupley`) and is pure Python/NumPy — no binary needed. So we now compute true SASA from the same CIF files already loaded, divide by the residue's max-ASA (Tien 2013 table already in the file) to get RSA, and only fall back to `1.0` if a residue truly can't be scored.

### New imports
```python
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.HSExposure import HSExposureCB
```

### New helper functions
- `_annotate_sasa_and_hse(structure)` — runs `ShrakeRupley().compute(model, level="R")` and `HSExposureCB(model)` **once per structure** (guarded by a `_sasa_hse_done` flag so the cached structure isn't recomputed). Computed on the whole model ⇒ quaternary (interface) burial is captured.
- `_rsa_from_sasa(residue)` — `residue.sasa / MAX_ASA[aa]`.
- `_hse_from_residue(residue)` — reads `EXP_HSE_B_U` / `EXP_HSE_B_D` from `residue.xtra`.

### Integration in `extract_structural_features`
```python
structure = _structure_cache[pdb_id]
_annotate_sasa_and_hse(structure)   # <-- added, runs once per PDB
...
# RSA: prefer mkdssp ASA if present, else Shrake-Rupley, else impute 1.0
if np.isnan(rsa):
    rsa = _rsa_from_sasa(residue)   # <-- added line
if np.isnan(rsa):
    rsa = 1.0
```

### Result of the fix (per-column, NEW code)
```
rsa   min=0.00 max=1.00 mean=0.36 unique=137   <-- was 1 unique value
```
Mean RSA ≈ 0.36 means most mapped residues are partially buried — sensible for a membrane protein.

## 5. Change 3 — half-sphere exposure features

**File:** `vep_nachr/features/structural.py`

Added two features, `hse_up` and `hse_down` (half-sphere exposure, Hamelryck 2005). For each residue, BioPython counts how many Cβ neighbours fall in the "upper" vs "lower" half-sphere defined by the Cα–Cβ direction. It's a cheap, robust burial/directionality measure that complements RSA (RSA = surface area; HSE = directional neighbour count).

```python
hse_up, hse_down = _hse_from_residue(residue)
...
features[i] = [rsa, bfactor, helix, sheet, coil, cbeta, hse_up, hse_down]
```

Names + imputation updated accordingly, and a module constant `N_STRUCTURAL_FEATURES = len(get_structural_feature_names())` was added so the matrix width is derived in one place instead of hard-coded `6`.

Observed ranges (NEW): `hse_up` 0–31 (mean 11.8), `hse_down` 0–26 (mean 12.8).

> Note: like `cbeta_density`, HSE and SASA are left **un-normalized** (raw counts / Å²). Tree models (RF, LightGBM, XGBoost) are scale-invariant; the scaled models (LogReg, SVM, KNN, MLP) get `RobustScaler` in the CV pipeline, so this is consistent with the existing convention.

## 6. Change 4 — housekeeping

- **`vep_nachr/config.py`** — `FEATURE_GROUPS['structural']` updated `n_features` 6 → 8 and refreshed the description (adds half-sphere exposure). Used for ablation bookkeeping only.
- **`vep_nachr/features/encoder.py`** — corrected the module docstring counts: structural 6→8, subunit one-hot 15→16, totals ‘49/55’ → ‘44 (no structural) / 52 (with structural)’. (Docstring only; the encoder already computed names dynamically.)

## 7. File-by-file summary

| File | Change | Type |
|---|---|---|
| `.claude/settings.local.json` | removed invalid `mcp__*` allow rule | config |
| `vep_nachr/features/structural.py` | Shrake-Rupley RSA + HSE features + helpers + `N_STRUCTURAL_FEATURES` | **feature logic** |
| `vep_nachr/config.py` | structural `n_features` 6→8 + desc | bookkeeping |
| `vep_nachr/features/encoder.py` | docstring feature counts corrected | docs |
| `VEP Nachr/NOTES2.ipynb` | this change log | docs |

No data files, no model registry, no cross-validation logic were changed. The public API of `extract_structural_features` is unchanged (still returns an `(n, k)` array, now `k=8`); the encoder picks up the new width and names automatically.

## 8. Results: before vs after

Quick mode (5-fold CV × 5 seeds, default hyperparameters, no Optuna). "Before" = structural with dead RSA (6 feats); "After" = Shrake-Rupley RSA + HSE (8 feats).

Measured (`--quick`, default HPs):

| Model | F1 no-struct | F1 struct (before, dead RSA) | F1 struct (after, real RSA+HSE) | Δ vs before |
|---|---|---|---|---|
| **logistic_regression** | 0.648 | 0.653 | **0.667** | **+0.014** |
| svm_rbf | 0.600 | 0.617 | 0.616 | -0.001 |
| random_forest | 0.635 | 0.627 | 0.608 | -0.019 |
| lightgbm | 0.654 | 0.666 | 0.659 | -0.007 |

**Read this honestly:**
- The **best overall F1 moved from 0.666 (LightGBM) to 0.667 (LogReg)** — essentially flat at the headline level.
- The **scaled linear model (LogReg) gained the most** (+0.014, and is now the single best model) because real-valued RSA/HSE give it a smooth burial signal it can weight.
- **Tree models dipped slightly** at *default* hyperparameters. This is expected and not alarming: (a) `--quick` does no HP tuning, and adding features changes the optimal tree depth / feature-subsampling, so the fixed defaults are now slightly mis-set; (b) random_forest has a large std (±0.09) so its -0.019 is within noise.
- The real value of the fix is that **RSA is now genuine signal instead of a constant**. The fair comparison is the full Optuna run (`python scripts/run_experiment.py`), which re-tunes each model for the new feature set — that is where the structural signal should pay off, especially after more subunits get structures (§10).

**Next measurement to run:** full nested-CV with Optuna on the 52-feature set, plus a feature-ablation (drop the structural block) to isolate its contribution per model.

---

### 8b. Full Optuna run — TUNED results (added this session)

This is the fair comparison promised above: full nested CV (5 outer × 5 inner folds × 5 seeds × 50 Optuna trials per fold) on the **52-feature** engineered set (with the new Shrake-Rupley RSA + HSE), each model re-tuned for the new feature set.

| Model | F1 (GOF) | Std | Accuracy |
|---|---|---|---|
| **logistic_regression** | **0.6595** | 0.0118 | 0.7197 |
| lightgbm | 0.6564 | 0.0170 | 0.7397 |
| svm_rbf | 0.6501 | 0.0290 | 0.7249 |
| random_forest | 0.6281 | 0.0114 | 0.7038 |

**Verdict (honest):**
- Tuned best F1 = **0.660 (LogReg)**, vs the documented pre-fix baseline of 0.666 (LightGBM). The structural fix did **not** move tuned F1 outside the ±0.01–0.03 noise band.
- RSA is now mechanically *alive* (137 unique values vs 1), but alive != predictive: at 351 samples the structural block doesn't add signal the physicochemical + substitution features don't already carry.
- **random_forest is both the slowest (~4 h for one model) and the weakest (0.628)** — a candidate to drop from the core set.
- 84/351 rows (24%) still get **imputed** structural features (6 subunits have no structure) — this dilutes any real structural signal. Adding AlphaFold models (section 10) is the highest-leverage next step before judging structural features as dead weight.
- **The clean test still outstanding:** a structural-block ablation (run the same 4 models on 44 features, structural OFF) to isolate the contribution head-to-head. If 44-feat F1 ~ 52-feat F1, structural is confirmed neutral.

**Engineering note:** the full run was initially a no-op because **Optuna was not installed** — every model silently failed the `if not OPTUNA_AVAILABLE` guard and produced nothing. Installing `optuna` (4.9.0) fixed it. Also parallelized the Optuna trials across all 16 cores (`cross_validation.py`: `study.optimize(..., n_jobs=os.cpu_count())`, each trial single-threaded) — the run previously used only ~5 of 16 cores.

## 9. Candidate structural features to add next

All of these are computable from the CIF coordinates we already have, with **no external binaries** (Windows-friendly). Ranked by expected value for nAChR LOF/GOF:

### Tier 1 — ion-channel-specific geometry (highest value, unique to this protein)
1. **Distance to the pore axis (radial position).** A nAChR is a pentamer with the ion pore on its 5-fold symmetry axis. Pore-lining residues (especially TM2) directly set conductance and gating — the textbook GOF/LOF hotspots. Compute the assembly's principal axis (membrane normal) via PCA on Cα coordinates, then take each residue's perpendicular distance to that axis. *Small radius = pore-facing.*
2. **Axial position along the pore axis (membrane depth).** Project the residue's Cα onto the symmetry axis. This gives a *quantitative* ECD / TMD / ICD coordinate — far better than the current crude `position_normalized` (which is just sequence index / max).

### Tier 2 — interface & packing
3. **Distance to nearest neighbouring subunit (interface proximity).** Min distance from the residue to any atom of a *different* chain. The ACh binding site and many gating residues sit at subunit interfaces.
4. **Multi-radius contact number.** Generalize `cbeta_density`: heavy-atom neighbour counts at e.g. 8 Å and 12 Å, optionally split into same-chain vs cross-chain. Richer packing/burial signal.

### Tier 3 — local conformation / chemistry
5. **Backbone dihedrals (φ/ψ) or a Gly/Pro-in-strained-context flag.** Captures helix-breaking / backbone-strain mutations (Pro into TM2 = classic disruptor).
6. **Proximity to a Cys-loop disulfide.** Cys-loop receptors have signature disulfides; mutations near them disrupt folding.
7. **Distance to the orthosteric (ACh) binding site.** Roadmap Priority 2 #6. Approximate via the conserved aromatic-box residues if no ligand is present in the structure.

**My recommendation:** implement **#1 + #2 (pore-axis radial + axial)** first — they encode the exact geometry that governs channel function and are not captured by any current feature. Then **#3 (interface distance)**. I can wire these in the same `structural.py` pattern (cached per structure) when you give the go-ahead.

**Caveat:** the pore-axis features require the *assembled pentamer*, so they only apply to the experimental cryo-EM structures (7QKO, 7EKI, 6CNJ, 6PV7). AlphaFold *monomer* models won't support them (no quaternary context), though they still support RSA / DSSP / HSE / contacts.

## 10. Do we need more protein structures?

**Short answer: yes, it would help — but it's not required to run.** Right now 84/351 mutations get imputed structural features. ~40 of those are because **6 subunits have no structure at all**:

| Subunit | UniProt | Mutations imputed for lack of structure |
|---|---|---|
| CHRNA6 | Q15825 | 18 |
| CHRNA2 | Q15822 | 9 |
| CHRNA5 | P30532 | 5 |
| CHRNB3 | Q05901 | 4 |
| CHRNA9 | Q9UGM1 | 2 |
| CHRNA10 | Q13002 | 2 |

The remaining ~44 imputed are positions not resolved in the existing cryo-EM structures (flexible ICD loops, termini).

### Recommendation
1. **Download AlphaFold models** for those 6 subunits from the AlphaFold DB (free, by UniProt ID, e.g. `https://alphafold.ebi.ac.uk/files/AF-Q15825-F1-model_v4.cif`). Drop the CIFs in `data/raw/structure_files/` and add entries to `PDB_MAPPING` in `config.py`. This recovers ~40 mutations and, because AlphaFold models are full-length, will also map some currently-unresolved positions.
2. **Optional upgrades** for the existing covered subunits: newer experimental α9α10 / α3α4 / α4α2 cryo-EM structures exist if you want true B-factors instead of imputation in flexible regions.

### Two caveats to document when adding AlphaFold
- AlphaFold's per-atom **B-factor field is pLDDT (model confidence), not thermal mobility.** So the `bfactor` feature changes meaning for those subunits. Either (a) accept it as a "confidence/disorder" proxy, or (b) zero it out for AlphaFold-derived rows and flag with an `is_alphafold` indicator.
- AlphaFold gives **monomers**, so RSA from a monomer over-estimates exposure at interfaces, and the proposed pore-axis features (§9 #1–2) don't apply. For exposure consistency you'd ideally assemble the predicted subunit into the pentamer, but a monomer is a reasonable first pass.

**Bottom line:** I don't *need* more structures to keep going, but AlphaFold models for the 6 uncovered subunits are the single highest-leverage data addition for the structural features — say the word and I'll script the download + `PDB_MAPPING` wiring.

## 11. Other (non-structural) features we could add

Section 9 covered structural features. These are the **non-structural** levers — several are likely higher-value than more structural features, because the tuned results (section 8b) suggest the current structural block is near its signal ceiling.

### Tier 1 — evolutionary / conservation (usually the single biggest win for VEP)
1. **Per-position conservation** from a multiple sequence alignment (MSA) of nAChR orthologs/paralogs — e.g. Shannon entropy or a position-specific scoring matrix (PSSM) column. Mutations at conserved sites are far more likely to be functional. This is the feature most VEP tools (SIFT, PolyPhen) lean on and we currently have **none** of it.
2. **Substitution likelihood from the PSSM** — score the specific wt->mut change against the alignment column, not just BLOSUM62 (which is position-agnostic).
3. **Grantham/Miyata deviation from the column consensus** — how chemically odd the variant is *relative to what that position tolerates*.

### Tier 2 — embeddings from a protein language model (pLM)
4. **ESM-2 / ESM-1v per-residue embeddings** (or the zero-shot variant-effect log-likelihood ratio `log p(mut) - log p(wt)`). These implicitly encode conservation + structure + biophysics learned from millions of sequences and are state-of-the-art for VEP. One forward pass per subunit sequence; no MSA needed. Heavier dependency, but the most likely thing to actually raise F1.

### Tier 3 — domain / annotation context
5. **Domain / topology label** (ECD vs TM1-4 vs ICD, loop C, Cys-loop, M2 pore-lining) from UniProt features — a cleaner categorical than raw position. Cheap and interpretable.
6. **Distance (in sequence) to known functional motifs** — the vicinal Cys pair, the M2 9'/leucine gate, the principal/complementary binding-site loops.

### Tier 4 — data-quality / meta features (use with care)
7. **Subunit-level priors** are already partly captured by the one-hot, but an explicit `is_muscle_type` / `is_homomeric_alpha7` flag can help small models.
8. **Measurement-technique flag** (already in the raw data as `technique`) — *use only if you're predicting biology, not assay artifacts*; risks leakage, so I'd keep it out of the published model and only test it as a diagnostic.

**My recommendation:** add **#1-#3 (MSA conservation/PSSM)** first — biggest expected F1 gain, no heavy dependencies, and we already have the 16 wildtype FASTAs to build alignments from. If you want to go state-of-the-art, **#4 (ESM-2 LLR)** is the highest ceiling. Both are far more promising than squeezing more structural features out of partial cryo-EM coverage.

## 12. ENaC comparison (sibling VEP-Enac project)

There is a sibling project in the same repo, `VEP-Enac/`, that does the same kind of variant-effect prediction for the **Epithelial Sodium Channel (ENaC)**. I ran it and compared to our nAChR results.

**Command run** (apples-to-apples on the feature side — same 4 core models, same nested CV depth, the `engineered` encoding = AAIndex + BLOSUM62 + structural, default `cleaned_nodup` dataset):
```
python scripts/experiments/run_comparison.py     --models logistic_regression svm_rbf random_forest lightgbm     --encodings engineered --data-sources cleaned_nodup --n-jobs -1
```

### Raw results

**ENaC** — 322 samples, 37 features, 3-class, **macro-F1**:

| Model | macro-F1 | Std |
|---|---|---|
| **lightgbm** | **0.5096** | 0.0196 |
| random_forest | 0.4960 | 0.0125 |
| svm_rbf | 0.4934 | 0.0089 |
| logistic_regression | 0.4919 | 0.0230 |

**nAChR (ours)** — 351 samples, 52 features, binary, **F1 on GOF**:

| Model | F1 (GOF) | Std |
|---|---|---|
| **logistic_regression** | **0.6595** | 0.0118 |
| lightgbm | 0.6564 | 0.0170 |
| svm_rbf | 0.6501 | 0.0290 |
| random_forest | 0.6281 | 0.0114 |

### ⚠️ The numbers are NOT directly comparable

This is the most important thing to understand before reading anything into "0.66 > 0.51":

| | nAChR (ours) | ENaC (sibling) |
|---|---|---|
| Task | **binary** (LOF vs GOF) | **3-class** (loss / no-net / gain) |
| Metric | F1 on the positive (GOF) class | **macro**-F1 averaged over 3 classes |
| Random-guess floor | ~0.50 | ~0.33 |
| Classes | 218 / 133 | 53 / 157 / 112 |
| Hard middle class? | no | **yes** — a "no-net-effect" class, which is intrinsically the hardest to separate |

A 3-class macro-F1 of 0.51 and a binary positive-class F1 of 0.66 can represent **similar real skill** once you account for the lower random-guess floor and the extra neutral class in ENaC. The ENaC project's own README reports the same ballpark (best ~0.538 with Random-Forest + structural), so our run reproduces their published level.

### What *is* fair to conclude
- **Both projects sit in the same regime:** small, imbalanced, biologically-driven datasets where engineered features get models to ~0.5–0.66 and no single model dominates. Neither is "solved."
- **Model ranking differs by task:** LightGBM tops ENaC; LogReg (barely) tops nAChR — but in both, the top 3 models are within noise of each other. Random-forest is consistently mediocre-to-slow in both.
- **The structural features behave the same way in both projects:** present but not decisive. ENaC's own ablation (their paper) found *no significant difference* between domain-driven (structural) and data-driven encodings (p=0.80) — independent confirmation of what we saw on the nAChR side in section 8b.

### To make it a true apples-to-apples comparison (optional next step)
Either (a) **binarize ENaC** (drop/merge the `no-net` class) and re-score with binary F1, or (b) **re-score nAChR with macro-F1**. Option (a) is the cleaner match to our task. Say the word and I'll run it.